# 06 — QSE group structure and inter-group bridges

In silicon burning the Si-group nuclei come into quasi-equilibrium among
themselves while flow *between* groups is throttled by a few **bridge**
reactions. Those bridges are where the physics (and the loss weighting)
lives. This node builds the group masks and ranks inter-group carriers from
the measured fluxes.

**The Si-group boundary is a config variant** (`configs/qse_groups.yaml`),
not a settled fact: default is 24 ≤ A < 45, `a24_46` puts the boundary at
A = 46 so ⁴⁵Sc sits *inside* the group. Every group/bridge figure here shows
**both** — a verdict that flips between them is a measured decision
(ADR 0004 revisit clause).

Exploratory only — citable values are the RESULTS.md 2026-07-10/12 rows.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd()
if not (_here / "nbsupport.py").exists():
    _root = next(p for p in [_here, *_here.parents] if (p / "pyproject.toml").exists())
    _here = _root / "notebooks" / "phase0"
sys.path.insert(0, str(_here))

import matplotlib.pyplot as plt
import numpy as np

import nbsupport as nbs
from gnn_nucleo.killtest.active_set import topk_concentration
from gnn_nucleo.killtest.manifold import assemble
from gnn_nucleo.killtest.strata import assign_strata
from gnn_nucleo.qse import load_group_mask

nbs.style()
QUICK = nbs.QUICK
NETS = ["mesa_80", "mesa_151"]
VARIANTS = ("default", "a24_46")
MAX_ROWS = 8 if QUICK else 24
RUN_IDS = ["trajectories-unscreened"] + ([] if QUICK else ["rerun-trajectories-unscreened"])

In [ ]:
nbs.provenance_header(
    "06",
    "QSE group structure + inter-group bridges",
    nbs.status_of([6]),
    results_rows=[
        "2026-07-10: ⁴⁵Sc(p,γ)⁴⁶Ti preliminarily CONFIRMED — rank 2/251 inter-group carriers on relaxed high-Yₑ QSE-window rows (a24_46 variant, mesa_151)",
        "2026-07-12: bridge sets FINALIZED (Step 6) — mesa_80: Ne22(α,n)Mg25 / Al27(p,α)Mg24 / P31(p,α)Si28; mesa_151 incl. Ca44(p,γ)Sc45 feeder; ⁴⁵Sc(p,γ)⁴⁶Ti rank 2/251 on the full relaxed manifold",
        "2026-07-12: r_QSE single-cluster plateau NOT confirmed on clean data (0/303, 0/316 rows < 0.1 dex) — Step-5 deferral retired with a negative verdict",
    ],
    data=[
        "data/fluxes/{net}/trajectories-unscreened (+ rerun-trajectories-unscreened at NB_QUICK=0)",
        "configs/qse_groups.yaml (group-boundary variants)",
    ],
    scripts=["scripts/step5_bridges.py", "scripts/run_killtest.py --relaxed --include-reruns"],
)

## Figure 1 — the Si group under both boundary conventions

Group membership on the (N, Z) plane. The only difference between the
variants is whether A = 45 (notably ⁴⁵Sc) counts as inside the group — which
decides whether ⁴⁵Sc(p,γ)⁴⁶Ti is a *bridge* (a boundary crossing) or an
*internal* group reaction.

In [ ]:
from gnn_nucleo.graph.isotopes import load_isotope_table  # lazy: pynucastro once

fig, axes = plt.subplots(2, 2, figsize=(12.5, 8.5), sharex=True, sharey=True)
for i, net in enumerate(NETS):
    tab = load_isotope_table(net)
    Z = np.asarray(tab.Z, float)
    A = np.asarray(tab.A, float)
    for j, variant in enumerate(VARIANTS):
        ax = axes[i, j]
        g = load_group_mask(net, variant)
        ax.scatter((A - Z)[~g], Z[~g], s=28, c="#CCCCCC", label="outside group", edgecolors="none")
        ax.scatter((A - Z)[g], Z[g], s=28, c="#0072B2", label=f"Si group (n={int(g.sum())})",
                   edgecolors="none")
        if "sc45" in tab.names:
            k = list(tab.names).index("sc45")
            ax.scatter([(A - Z)[k]], [Z[k]], s=170, facecolors="none",
                       edgecolors="#D55E00", linewidths=2.2)
            ax.annotate(f"sc45 {'IN' if g[k] else 'OUT'}", ((A - Z)[k], Z[k]),
                        textcoords="offset points", xytext=(10, -20), fontsize=8.5,
                        color="#D55E00", fontweight="bold")
        ax.set_title(f"{net} · variant '{variant}'", fontsize=10)
        if i == 1:
            ax.set_xlabel("N = A − Z")
        if j == 0:
            ax.set_ylabel("Z")
axes[0, 0].legend(fontsize=8, loc="upper left")
fig.suptitle("Si-group membership — BOTH boundary variants (ADR 0004: the boundary is a config variant)",
             y=1.0)
nbs.caption(
    fig,
    "The variants differ only at A = 45. mesa_80 has no ⁴⁵Sc at all (its only Sc is ⁴³Sc), so the "
    "⁴⁵Sc bottleneck instrument is mesa_151-only — one of the asymmetries that makes size transfer "
    "a hypothesis rather than an assumption.",
    results=[
        "RESULTS.md 2026-07-08 isotope-list rows (sc45 absent from mesa_80)",
        "docs/decisions/0004 (Si-group boundary as a config variant; revisit clause)",
    ],
    scripts=["configs/qse_groups.yaml"],
)

## Load relaxed rows (no NSE solves)

`assemble(..., with_delta=False)` skips the Saha solves — this notebook only
needs fluxes and coordinates. QUICK: shipped trajectories, 8 rows each.
The measured bridge rankings used the full relaxed manifold (20 shipped +
209 rerun trajectories/net, 24 rows each).

In [ ]:
rows = {}
for net in NETS:
    r = assemble(net, RUN_IDS, prestall=True, max_rows_per_traj=MAX_ROWS, with_delta=False)
    t9b, yeb = assign_strata(r.T, r.ye)
    hi = (yeb == 2) & ((t9b == 2) | (t9b == 3))  # high Yₑ, QSE window (3.3–5 GK)
    rows[net] = (r, hi)
    print(f"{net}: {r.n_rows} rows from {len(set(r.traj_key))} trajectories; "
          f"{int(hi.sum())} high-Yₑ QSE-window rows")

## Figure 2 — inter-group carriers, both variants side by side

A reaction *crosses the group boundary* when it changes the group's total
mass: `ag = (A · g) @ ν` is nonzero. Ranking by
`Σ_rows |ag_j · φ_j|` over high-Yₑ QSE-window rows gives the empirical
bridge set. The ranking below MIRRORS `scripts/run_killtest.py` §3 (same
`ag`/share definition, same high-Yₑ QSE-window row selection) — it is a
re-implementation for plotting, so the script remains the authority for any
citable rank.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for i, net in enumerate(NETS):
    r, hi = rows[net]
    z = nbs.load_nu(net)
    nu, rate_strings, rate_fnames = z["nu"], z["rate_strings"], z["rate_fnames"]
    A = np.asarray(load_isotope_table(net).A, float)
    from gnn_nucleo.fluxes.store import FluxStore

    c0 = next(iter(FluxStore(net, RUN_IDS[0]).iter_chunks()))
    net_cols = c0.is_forward_member | (c0.pair_col < 0)

    for j, variant in enumerate(VARIANTS):
        ax = axes[i, j]
        g = load_group_mask(net, variant)
        ag = (A * g) @ nu
        xc = np.nonzero((np.abs(ag) > 0) & net_cols)[0]
        if not hi.any() or xc.size == 0:
            ax.set_visible(False)
            continue
        share = np.abs(ag[xc, None] * r.phi[xc][:, hi]).sum(axis=1)
        order = np.argsort(-share)
        top = order[:15]
        frac = share[top] / share.sum()
        y = np.arange(len(top))[::-1]
        labels = [str(rate_strings[xc[k]])[:40] for k in top]
        is_sc = ["Sc45" in str(rate_fnames[xc[k]]) and "Ti46" in str(rate_fnames[xc[k]])
                 for k in top]
        ax.barh(y, frac, color=["#D55E00" if s else "#0072B2" for s in is_sc])
        ax.set_yticks(y, labels, fontsize=6.5)
        tk = topk_concentration(share, (1, 5))
        ax.set_title(f"{net} · '{variant}' — {len(xc)} inter-group carriers\n"
                     f"top-1 {tk[1]:.2f}, top-5 {tk[5]:.2f} of |inter-group flow|", fontsize=9)
        ax.set_xlabel("share of Σ|ag·φ| over high-Yₑ QSE rows")
        if net == "mesa_151":
            sc = [k for k, s in enumerate(rate_fnames)
                  if "Sc45" in str(s) and "Ti46" in str(s) and str(s).startswith("p_")]
            for k in sc:
                hit = np.nonzero(xc[order] == k)[0]
                if hit.size:
                    print(f"{net} [{variant}]: ⁴⁵Sc(p,γ)⁴⁶Ti rank {int(hit[0]) + 1}/{len(xc)} "
                          f"(quick-look; measured rank 2/251 on the full manifold)")
fig.suptitle("Inter-group carriers ranked — BOTH boundary variants", y=1.0)
if QUICK:
    nbs.quick_banner(fig)
nbs.caption(
    fig,
    "Quick-look ranking on shipped-trajectory rows only; the FINALIZED sets were measured on the "
    "full relaxed manifold (20 shipped + 209 rerun trajectories/net): mesa_80's bridges are "
    "Ne22(α,n)Mg25 / Al27(p,α)Mg24 / P31(p,α)Si28, and ⁴⁵Sc(p,γ)⁴⁶Ti ranks 2/251 in mesa_151 "
    "under a24_46 — where ⁴⁵Sc is inside the group, so the reaction is a boundary crossing. Ranks "
    "here may differ from the measured ones (fewer rows). Both variants are shown because the "
    "boundary is a config choice, not a fact.",
    results=[
        "RESULTS.md 2026-07-12 bridge-set rows (finalized; ⁴⁵Sc(p,γ)⁴⁶Ti rank 2/251, a24_46)",
        "RESULTS.md 2026-07-10 ⁴⁵Sc preliminary-confirmation row",
    ],
    scripts=["scripts/run_killtest.py --relaxed --include-reruns", "scripts/step5_bridges.py"],
)

## Figure 3 — does the boundary choice change the answer?

The ADR-0004 revisit clause in action: if a verdict flips between variants,
that flip is itself a measured decision. Here: how concentrated is
inter-group flow under each convention?

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.4))
x = np.arange(len(NETS) * len(VARIANTS))
vals, labels = [], []
for net in NETS:
    r, hi = rows[net]
    z = nbs.load_nu(net)
    A = np.asarray(load_isotope_table(net).A, float)
    from gnn_nucleo.fluxes.store import FluxStore

    c0 = next(iter(FluxStore(net, RUN_IDS[0]).iter_chunks()))
    net_cols = c0.is_forward_member | (c0.pair_col < 0)
    for variant in VARIANTS:
        g = load_group_mask(net, variant)
        ag = (A * g) @ z["nu"]
        xc = np.nonzero((np.abs(ag) > 0) & net_cols)[0]
        share = np.abs(ag[xc, None] * r.phi[xc][:, hi]).sum(axis=1) if hi.any() else np.zeros(1)
        tk = topk_concentration(share, (1, 5, 10))
        vals.append([tk[1], tk[5], tk[10]])
        labels.append(f"{net.replace('mesa_', '')}\n{variant}")
vals = np.array(vals)
for k, (lab, c) in enumerate([("top-1", "#0072B2"), ("top-5", "#E69F00"), ("top-10", "#009E73")]):
    ax.bar(x + (k - 1) * 0.26, vals[:, k], 0.25, label=lab, color=c)
ax.set_xticks(x, labels, fontsize=8)
ax.set_ylabel("share of |inter-group flow|")
ax.set_ylim(0, 1.05)
ax.set_title("Concentration of inter-group flow by network × boundary variant")
ax.legend(fontsize=8)
if QUICK:
    nbs.quick_banner(fig)
nbs.caption(
    fig,
    "Inter-group flow is carried by a handful of reactions under either convention — the "
    "bottleneck picture is boundary-INSENSITIVE, which is why the kill-test verdict was recorded "
    "as verdict-insensitive to the group variant (the a24_46 variant is retained for ⁴⁵Sc "
    "analyses specifically). Quick-look shares; the measured concentrations are in the cited rows.",
    results=[
        "RESULTS.md 2026-07-12 top-k concentration / bridge rows",
        "docs/phase0-killtest-verdict.md Step-7 handoff (group boundary: verdict-insensitive)",
    ],
    scripts=["scripts/run_killtest.py --relaxed --include-reruns"],
)

## TODO (stub)

- **mesa_80 bridge set vs the Step-5 preliminary list** — Step 5 proposed
  the Ne22(α,n)Mg25 / Al27(p,α)Mg24 / P31(p,α)Si28 class; Step 6 finalized
  it on the full relaxed manifold. A side-by-side of preliminary vs final
  ranks would make the confirmation legible. Producer:
  `scripts/step5_bridges.py` + `scripts/run_killtest.py`.

## What this notebook does NOT show

- **Group equilibrium itself.** The r_QSE single-cluster plateau was NOT
  confirmed on clean data (0/303, 0/316 rows < 0.1 dex) — these groups are
  not measurably in QSE on this manifold, which is part of why the Guidry
  mask came out empty (notebook 10).
- δ_r / NSE departures: notebooks 09 (labels) and 10 (mask).